# Phase 1: Proof of Concept (PoC) - Data Sampling
This notebook focuses on loading the raw ticket dataset and extracting a strategic sample of 50 tickets (including edge cases like short, long and noisy texts) for our LLM prompt testing.

## 1. Environment Setup and Relative Paths
Importing required libraries and setting up relative paths to ensure portability across different environments.

In [34]:
import pandas as pd
import numpy as np
import re
import os

In [35]:
# Define paths and variables
CONFIG = {
    "data_dir" : "../data",
    "raw_data_file" : os.path.join("../data", "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"),
    "output_sample_file" : os.path.join("../data", "poc_sample_50.csv"),
    "text_col" : "response"
}

In [36]:
# Ensure the data directory exists
os.makedirs(CONFIG["data_dir"],exist_ok=True)

In [37]:
# Define a function to calculate a 'noise ratio'
def calculate_noise_ratio(text):
    text = str(text)
    if len(text) == 0:
        return 0
    # Count anything that is not a letter, number, or space
    special_chars = len(re.findall(r'[^a-zA-Z0-9\s]', text))
    return special_chars / len(text)

## 2. Data Loading
Loading the raw dataset to begin our strategic sampling process.

In [38]:
#  Load the dataset
try:
  df = pd.read_csv(CONFIG["raw_data_file"])

except FileNotFoundError:
  print(f"Error: Dataset not found at {CONFIG['raw_data_file']}")


## 3. Strategic Edge-case Sampling (Mini-batch)
Instead of purely random sampling, we extract 50 specific edge-cases:
- Very short tickets (Context starvation)
- Very long tickets (Token limit & summarization test)
- Noisy tickets (Robustness test)
- Normal/Average tickets (Baseline)

In [39]:
# check if dataframe is loaded before proceeding
if 'df' in locals() and not df.empty:
  # calulate text length
  df['text_length'] = df[CONFIG["text_col"]].astype(str).apply(len)

  # calculate noise score
  df['noise_score'] = df[CONFIG["text_col"]].apply(calculate_noise_ratio)

  # Sample based on edge-cases
  # A.Shortest tickets
  short_tickets = df[df['text_length']>10].sort_values('text_length').head(10)

  # B.Longest tickets
  long_tickets = df.sort_values('text_length', ascending=False).head(10)

  # C. Noisiest tickeets
  noisy_tickets = df.sort_values('noise_score',ascending= False).head(15)

  # D. Normal tickets(around median length)
  median_len = df['text_length'].median()
  median_noise = df['noise_score'].median()
  normal_tickets = df[(df['text_length']>median_len * 0.8) &
                      (df['text_length']<median_len * 1.2) &
                      (df['noise_score']<median_noise)].sample(15,random_state=42)

  # Combine, drop duplicates, and shuffle
  poc_sample = pd.concat([short_tickets, long_tickets, noisy_tickets, normal_tickets])
  poc_sample = poc_sample.drop_duplicates(subset=[CONFIG["text_col"]])
  poc_sample = poc_sample.sample(frac=1, random_state=42).reset_index(drop=True)

  # Trim the final count and warn if it's less than expected
  poc_sample = poc_sample.head(50)

  # Check the final count and warn if it's less than expected
  final_count = len(poc_sample)
  if final_count < 50:
    print(f"Warning: Expected 50 tickets, but only got {final_count}.")
  else:
    print(f"Successfully generated a PoC sample with {len(poc_sample)} unique tickets.")

Successfully generated a PoC sample with 50 unique tickets.


## 4. Exporting the PoC Mini-batch
Saving the generated sample to the data folder so it can be consumed by the prommpt engineering notebook.


In [40]:
if 'poc_sample' in locals():
  # Save the extracted sample
  poc_sample.to_csv(CONFIG["output_sample_file"],index=False)
  print(f"Sample dataset successfully saved to:{CONFIG["output_sample_file"]}")

Sample dataset successfully saved to:../data/poc_sample_50.csv
